In [29]:
import torch
from typing import List, Tuple
from copy import deepcopy
import numpy as np


def get_network_structure(
    spacing: List,  # (1,1)
    input_feature_size: List,  # (1024,512)
    ds_factor=2.0,  # down sample facotr
):
    """
    1. feature size at the last layer, >= 4 for each resolution dim
    2. downsample(scale=2.0) at the start layer of every stage
    3.
    """
    min_feature_size = 4
    dim = len(spacing)

    # 1. (h,w)
    current_size = deepcopy(list(input_feature_size))
    current_spacing = deepcopy(list(spacing))

    # 2. pool_kernels(list) has one more element than conv_kernels, so need to add one more at the end, see the following code below.
    pool_kernel_sizes = [[1] * len(spacing)]  # ([2,2], [2,2], [2,1], [2,1], [2,1], [1,1])
    conv_kernel_sizes = []  # ([3,3], [3,3], [3,3], [3,3])

    num_pool_ops = [0] * dim
    kernel_size = [1] * dim

    # 3. down sample
    assert ds_factor >= 1.0, "downsample factor should be > 1.0"

    # 4.
    while True:
        # feature size constraint
        valid_axis_for_pool = [i for i in range(dim) if current_size[i] > 2 * min_feature_size]

        #
        if len(valid_axis_for_pool) < 1:
            break

        # spacing constraint, why need ?
        valid_axis_spacing = [current_spacing[i] for i in valid_axis_for_pool]
        min_spacing = min(valid_axis_spacing)
        valid_axis_spacing = [sp for sp in valid_axis_spacing if sp / min_spacing < 2]

        # if only one axis left, corresponding feature size should be larger than 3*min_feature size
        if len(valid_axis_for_pool) == 1:
            if current_size[valid_axis_for_pool[0]] >= 3 * min_feature_size:
                ...
            else:
                break
        elif len(valid_axis_for_pool) < 1:
            break

        for i in range(dim):
            if kernel_size[i] == 3:
                continue
            elif current_spacing[i] / min(current_spacing) < 2:  # why need?
                kernel_size[i] = 3

        other_axis = [i for i in range(dim) if i not in (valid_axis_for_pool)]

        pool_size = [0] * dim
        for ax in valid_axis_for_pool:
            pool_size[ax] = 2
            num_pool_ops[ax] += 1
            current_spacing[ax] *= 2
            current_size[ax] = np.ceil(current_size[ax] / 2)

        for ax in other_axis:
            pool_size[ax] = 1

        pool_kernel_sizes.append(pool_size)
        conv_kernel_sizes.append(deepcopy(kernel_size))

    # 5. input feature should be
    should_be_divided_by = [2**i for i in num_pool_ops]
    # 6. padding
    padded_size = deepcopy(list(input_feature_size))
    for i in range(dim):
        mul = input_feature_size[i] // should_be_divided_by[i]
        padded_size[i] = (mul + 1) * should_be_divided_by[i]

    # 7. add one more conv kernel
    conv_kernel_sizes.append([3] * dim)
    return pool_kernel_sizes, conv_kernel_sizes, num_pool_ops, should_be_divided_by, padded_size


get_network_structure((1, 1), (1779, 131), 2)

([[1, 1], [2, 2], [2, 2], [2, 2], [2, 2], [2, 2], [2, 1], [2, 1], [2, 1]],
 [[3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3]],
 [8, 5],
 [256, 32],
 [1792, 160])

In [21]:
from copy import deepcopy


def get_pool_and_conv_props(spacing, patch_size, min_feature_map_size, max_numpool):
    """
    this is the same as get_pool_and_conv_props_v2 from old nnunet

    :param spacing:
    :param patch_size:
    :param min_feature_map_size: min edge length of feature maps in bottleneck
    :param max_numpool:
    :return:
    """
    # todo review this code
    dim = len(spacing)

    current_spacing = deepcopy(list(spacing))
    current_size = deepcopy(list(patch_size))

    pool_op_kernel_sizes = [[1] * len(spacing)]
    conv_kernel_sizes = []

    num_pool_per_axis = [0] * dim
    kernel_size = [1] * dim

    while True:
        # exclude axes that we cannot pool further because of min_feature_map_size constraint
        valid_axes_for_pool = [i for i in range(dim) if current_size[i] >= 2 * min_feature_map_size]
        if len(valid_axes_for_pool) < 1:
            break

        spacings_of_axes = [current_spacing[i] for i in valid_axes_for_pool]

        # find axis that are within factor of 2 within smallest spacing
        min_spacing_of_valid = min(spacings_of_axes)
        valid_axes_for_pool = [i for i in valid_axes_for_pool if current_spacing[i] / min_spacing_of_valid < 2]

        # max_numpool constraint
        valid_axes_for_pool = [i for i in valid_axes_for_pool if num_pool_per_axis[i] < max_numpool]

        if len(valid_axes_for_pool) == 1:
            if current_size[valid_axes_for_pool[0]] >= 3 * min_feature_map_size:
                pass
            else:
                break
        if len(valid_axes_for_pool) < 1:
            break

        # now we need to find kernel sizes
        # kernel sizes are initialized to 1. They are successively set to 3 when their associated axis becomes within
        # factor 2 of min_spacing. Once they are 3 they remain 3
        for d in range(dim):
            if kernel_size[d] == 3:
                continue
            else:
                if current_spacing[d] / min(current_spacing) < 2:
                    kernel_size[d] = 3

        other_axes = [i for i in range(dim) if i not in valid_axes_for_pool]

        pool_kernel_sizes = [0] * dim
        for v in valid_axes_for_pool:
            pool_kernel_sizes[v] = 2
            num_pool_per_axis[v] += 1
            current_spacing[v] *= 2
            current_size[v] = np.ceil(current_size[v] / 2)
        for nv in other_axes:
            pool_kernel_sizes[nv] = 1

        pool_op_kernel_sizes.append(pool_kernel_sizes)
        conv_kernel_sizes.append(deepcopy(kernel_size))
        # print(conv_kernel_sizes)

    def _to_tuple(lst):
        return tuple(_to_tuple(i) if isinstance(i, list) else i for i in lst)

    # we need to add one more conv_kernel_size for the bottleneck. We always use 3x3(x3) conv here
    conv_kernel_sizes.append([3] * dim)
    return (
        num_pool_per_axis,
        _to_tuple(pool_op_kernel_sizes),
        _to_tuple(conv_kernel_sizes),
        tuple(patch_size),
        current_spacing,
    )


spacing = [1, 1]
patch_size = [1997, 131]
min_feature_size = 4
max_numpool = 9999

get_pool_and_conv_props(spacing, patch_size, min_feature_size, max_numpool)

([8, 5],
 ((1, 1), (2, 2), (2, 2), (2, 2), (2, 2), (2, 2), (2, 1), (2, 1), (2, 1)),
 ((3, 3), (3, 3), (3, 3), (3, 3), (3, 3), (3, 3), (3, 3), (3, 3), (3, 3)),
 (1997, 131),
 [256, 32])

In [24]:
import numpy as np


np.mod(5, 3), 5 % 3, 5 // 3

(np.int64(2), 2, 1)

In [37]:
a = [111, 1, 1, 1, 1]

b = [a[i] for i in range(10) if i < len(a)]
b += [4] * (4 - len(a))

b, [4] * (-1)

([111, 1, 1, 1, 1], [])

In [43]:
a = [i for i in range(4, -1, -1)]
b = [1, 3]

[i for i in zip(a, b)]

[(4, 1), (3, 3)]

In [54]:
a = [
    0,
    1,
    2,
    3,
    4,
]
a[-2::-1], a[1:-1][::-1]

([3, 2, 1, 0], [3, 2, 1])

In [3]:
import os

name = "D:/ddd/sss.png"
os.path.splitext(name)[0] + ".jpg"

'D:/ddd/sss.jpg'

In [3]:
import torch


a = [torch.tensor([1, 2, 3]), torch.tensor([4, 5, 6])]

torch.stack(a, dim=1)

tensor([[1, 4],
        [2, 5],
        [3, 6]])